In [ ]:
import shapefile
import geopandas as gpd
import numpy as np

from matplotlib.cm import get_cmap

from scipy.ndimage import gaussian_filter1d
from sklearn.preprocessing import MinMaxScaler
from scipy.interpolate import splprep, splev
from shapely.geometry import LineString
from scipy.interpolate import interp1d

import torch
from torch_geometric.data import Data
from scipy.spatial import Delaunay

import matplotlib.pyplot as plt
import networkx as nx
from torch_geometric.utils import to_networkx

from shapely.geometry import shape as shapely_shape

In [ ]:
def plot_lines(lines, lines_syn_1=None, lines_syn_2=None, start=0, end=100, figsize=(10, 6), title=None):
    """
    Plots a number of 2D lines (shape: [N, num_points, 2])

    Args:
        lines (np.ndarray): Shape (N, num_points, 2)
        num_to_plot (int): How many lines to plot
        figsize (tuple): Size of the figure
    """
    plt.figure(figsize=figsize)

    for i in range(start,end):
        line = lines[i]
        x, y = line[:, 0], line[:, 1]
        #plt.scatter(x, y, color='#00305D', label='Original Line', s=20)
        plt.plot(x, y, color='#00305D', label=f'Original Line')

        if lines_syn_1 is not None:
            line_s1 = lines_syn_1[i]
            x_s1, y_s1 = line_s1[:, 0], line_s1[:, 1]
            plt.plot(x_s1, y_s1, color='#EC9A29', label=f'Synthetic Original')

        if lines_syn_2 is not None:
            line_s2 = lines_syn_2[i]
            x_s2, y_s2 = line_s2[:, 0], line_s2[:, 1]
            plt.plot(x_s2, y_s2, color='#A8201A', label=f'Synthetic Displaced')

        if title:
            plt.title(f'{title} - Line number {i}')
        else:
            plt.title(f'Line number {i}')
        
        plt.xlabel('X')
        plt.ylabel('Y')
        plt.axis('equal')
        plt.grid(True)
        plt.legend()
        plt.show()

## Load and Plot input Shapefile

In [ ]:
PATH_SHP = '../data/original/waterways_merged_reprojected.shp'

# Load as GeoDataFrame
gdf = gpd.read_file(PATH_SHP)
gdf = gdf.to_crs(epsg=3857)  # project to metric (optional, useful for distance ops)
print(gdf.shape)

# # Plot the shapefile
gdf.plot(figsize=(10, 10), edgecolor='#143642', linewidth=0.5)
plt.title("Rivers Regular Lines")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True)
plt.show()

In [ ]:
#Just converts input shape into np array
coords_list = [np.array(geom.coords) for geom in gdf.geometry]

In [ ]:
# def read_line_shp(path, num_points):
#   ''' Reads a shapefile and converts into array'''

#   with shapefile.Reader(path) as sf:
#     shapes = sf.shapes() #Retrieves all geometric shapes (lines) from the shapefile.
#     print("Original Shapes", len(shapes))
#     sublines = []
#     for shape in shapes: #Iterates through each shape
#       num_lines = len(shape.points) // num_points   # Calculates how many full num_points-point sublines can be extracted from the shape's points
#       #print("Number of segments within one line", num_lines)
#       i = num_points
#       for _ in range(num_lines): #Loops through the points
#         subline_points = shape.points[i-num_points : i]
#         sublines.append(subline_points) #slices the shape into chunks of num_points points and addds each num_points-point subline to the sublines list
#         i += num_points

#   arr = np.array(sublines)
#   print("Shape of original array", arr.shape)

#   # Returns a 3D NumPy array of shape (n, num_points, 2), where n is the number of n-point sublines, and each point has (x, y) coordinates.
#   return arr

In [ ]:
# segment_length = 128
# original_lines = read_line_shp(PATH_SHP, num_points=segment_length)

## Fixed Length River Segments

In [ ]:
def interpolate_line(points, num_points):
    """
    Interpolates a line to `num_points` points using linear interpolation.
    """

    points = np.array(points)
    cumulative_distance = np.cumsum(np.sqrt(np.sum(np.diff(points, axis=0)**2, axis=1)))
    cumulative_distance = np.insert(cumulative_distance, 0, 0)

    f_interp = interp1d(cumulative_distance, points, axis=0, kind='linear')
    new_distances = np.linspace(0, cumulative_distance[-1], num_points)
    return f_interp(new_distances)

def read_line_shp_interpolated(path, num_points=64, min_points=32, max_points=256):
    '''Reads a shapefile and returns interpolated lines with fixed num_points each'''

    with shapefile.Reader(path) as sf:
        shapes = sf.shapes()
        print("Original shapes:", len(shapes))

        sublines = []

        # Filter out lines within the given range
        shapes = [shape for shape in shapes if max_points>= len(shape.points) >= min_points]

        for shape in shapes:
            pts = shape.points
            total_len = len(pts)

            num_full_segments = total_len // min_points
            remainder = total_len % min_points

            # Add full 64-point chunks
            for i in range(num_full_segments):
                segment = pts[i*min_points : (i+1)*min_points]
                interpolated = interpolate_line(segment, num_points)
                sublines.append(interpolated)

            # Handle leftover segment
            if remainder >= min_points:
                leftover = pts[-remainder:]
                interpolated = interpolate_line(leftover, num_points)
                sublines.append(interpolated)

            elif remainder > 0:
                # Pad backwards to get 32-point chunk
                padded_segment = pts[-(min_points):]
                interpolated = interpolate_line(padded_segment, num_points)
                sublines.append(interpolated)

    arr = np.array(sublines)
    print(f"Final shape: {arr.shape}  (n_lines, num_points, 2)")
    return arr


In [ ]:
# Interpolated Lines 
original_lines = read_line_shp_interpolated(PATH_SHP, num_points=64, min_points=32, max_points=256)

In [ ]:
start = 21
plot_lines(original_lines, None, None, start=start, end=start+1, title="Original Lines")

In [ ]:
# np.save(f'../data/preprocessing/interpolated/interpolated_lines.npy', original_lines)

## **Rotation**

In [ ]:
interpolated_lines = np.load(f'../data/preprocessing/interpolated/interpolated_lines.npy')

In [ ]:
print(interpolated_lines.min(), interpolated_lines.max())
print(np.abs(interpolated_lines).mean())

In [ ]:
def rotate_line_to_vertical_up(points):
    """
    Transforms a 2D line so that:
    - Start point is at (0, 0)
    - End point is at (0, L) where L > 0
    - Line is aligned vertically, increasing upward

    Parameters:
        points: Nx2 numpy array of (x, y) coordinates

    Returns:
        Nx2 numpy array of transformed points
    """
    p0 = points[0]
    p1 = points[-1]

    # Step 1: Translate start point to origin
    translated = points - p0
    vec = p1 - p0

    # Step 2: Compute rotation angle to align vec with vertical (0, 1)
    angle = np.arctan2(vec[1], vec[0])  # angle of the line
    rotation_angle = np.pi/2 - angle    # rotate line to point straight up

    # Step 3: Build rotation matrix
    cos_a = np.cos(rotation_angle)
    sin_a = np.sin(rotation_angle)
    R = np.array([[cos_a, -sin_a],
                  [sin_a,  cos_a]])

    # Step 4: Rotate all points
    rotated = translated @ R.T

    # Step 5: If final y-value is still negative, flip entire y-axis
    if rotated[-1, 1] < 0:
        rotated[:, 1] *= -1

    return rotated

In [ ]:
# Rotation with interpolated lines 
rotated_lines = np.array([
    rotate_line_to_vertical_up(line) for line in original_lines
])

# Roation with original shapefile lines 
#rotated_lines = [rotate_line_to_vertical_up(coords) for coords in coords_list]


In [ ]:
plot_lines(rotated_lines, None, None, start=start, end=start+1, title="Rotated Lines")

In [ ]:
# np.save(f'../data/preprocessing/rotated/rotated_lines.npy', rotated_lines)

## Creation Synthetic Lines

In [ ]:
interpolated_lines = np.load(f'../data/preprocessing/interpolated/interpolated_lines.npy')

In [ ]:
def resample_line(line, n_points):
    distances = np.cumsum(np.linalg.norm(np.diff(line, axis=0), axis=1))
    distances = np.insert(distances, 0, 0)
    total_length = distances[-1]
    uniform_distances = np.linspace(0, total_length, n_points)
    interp_func = interp1d(distances, line, axis=0)
    return interp_func(uniform_distances)

In [ ]:
def chaikin_smoothing(points, iterations=2):
    for _ in range(iterations):
        new_points = [points[0]]
        for i in range(len(points) - 1):
            p0, p1 = points[i], points[i + 1]
            Q = 0.75 * p0 + 0.25 * p1
            R = 0.25 * p0 + 0.75 * p1
            new_points.extend([Q, R])
        new_points.append(points[-1])
        points = np.array(new_points)
    return points

In [ ]:
def generate_offset_line(line, offset_distance, noise_std=10, smoothing_iters=3,
                         adaptive_scaling=True, curvature_sigma=8):
    n_points = len(line)

    # Compute tangent vectors
    dx = gaussian_filter1d(np.gradient(line[:, 0]), sigma=curvature_sigma)
    dy = gaussian_filter1d(np.gradient(line[:, 1]), sigma=curvature_sigma)

    norms = np.sqrt(dx**2 + dy**2) + 1e-8
    dx /= norms
    dy /= norms

    # Compute normal vectors
    nx = -dy
    ny = dx

    # Offset magnitudes (with optional curvature-based tapering)
    offsets = offset_distance + np.random.normal(0, noise_std, size=n_points)

    if adaptive_scaling:
        # Curvature: use derivative of angle
        angles = np.arctan2(dy, dx)
        curvature = np.abs(np.gradient(angles))
        curvature = gaussian_filter1d(curvature, sigma=3)
        curvature_factor = 1 / (1 + curvature * 50)  # scale factor suppressing sharp bends
        offsets *= curvature_factor

    # Apply offsets
    offset_x = nx * offsets
    offset_y = ny * offsets
    offset_line = line + np.stack([offset_x, offset_y], axis=1)

    # Optional smoothing to fix jitter
    for _ in range(smoothing_iters):
        offset_line = chaikin_smoothing(offset_line)

    return offset_line

In [ ]:
def is_valid_line(line, river, min_distance=5.0, loop_area_threshold=300.0, simplify_tolerance=5.0):

    line_geom = LineString(line)
    river_geom = LineString(river)

    # Reject if the line intersects itself
    if not line_geom.is_simple:
        return False

    # Reject if it's too close to the original river
    if line_geom.distance(river_geom) < min_distance:
        return False

    # Reject if simplified version is still too dense (indicating loops)
    simplified = line_geom.simplify(simplify_tolerance)
    if len(simplified.coords) > 2 * len(line):
        return False

    # Reject if area enclosed is unusually small (tight loops)
    area = line_geom.buffer(1.0).area
    if area < loop_area_threshold:
        return False

    return True

In [ ]:
def generate_street_lines_from_rivers(rivers, n_points=64, close_offset=40, far_offset=120, max_attempts=32):
    """
    Generate synthetic 'street' lines offset from input river lines.
    Works with both:
      - NumPy arrays of shape (n_lines, n_points, 2)
      - Lists of variable-length NumPy arrays [(Ni, 2), ...]
    """

    # --- Detect input type and standardize ---
    if isinstance(rivers, np.ndarray):
        # If already 3D (n_lines, n_points, 2)
        n_lines = rivers.shape[0]
        get_line = lambda i: rivers[i]
    elif isinstance(rivers, (list, tuple)):
        # If list of arrays of shape (Ni, 2)
        n_lines = len(rivers)
        get_line = lambda i: np.asarray(rivers[i])
    else:
        raise TypeError("Input 'rivers' must be a NumPy array or list of NumPy arrays.")

    # --- Prepare outputs ---
    close_lines = []
    far_lines = []

    # --- Main generation loop ---
    for i in range(n_lines):
        river = resample_line(get_line(i), n_points)

        # --- Close line ---
        for attempt in range(max_attempts):
            close_candidate = generate_offset_line(river, offset_distance=close_offset, noise_std=10)
            if is_valid_line(close_candidate, river, min_distance=4.0):
                close_line = resample_line(close_candidate, n_points)
                break
        else:
            print(f"Warning: Close street for line {i} may be invalid.")
            close_line = resample_line(close_candidate, n_points)

        close_lines.append(close_line)

        # --- Far line ---
        for attempt in range(max_attempts):
            far_candidate = generate_offset_line(river, offset_distance=far_offset, noise_std=6)
            if is_valid_line(far_candidate, river, min_distance=32.0):
                far_line = resample_line(far_candidate, n_points)
                break
        else:
            print(f"Warning: Far street for line {i} may be invalid.")
            far_line = resample_line(far_candidate, n_points)

        far_lines.append(far_line)

    # --- Return as NumPy arrays if all shapes match ---
    try:
        close_lines = np.stack(close_lines)
        far_lines = np.stack(far_lines)
    except ValueError:
        # Fallback to lists if shapes differ
        print("Note: Lines have varying lengths, returning lists instead of arrays.")
        pass

    return close_lines, far_lines


In [ ]:
# Full Synthetic Street Generator
# def generate_street_lines_from_rivers(rivers, n_points=64, close_offset=40, far_offset=120, max_attempts=32):
#     n_lines = rivers.shape[0]
#     close_lines = np.zeros((n_lines, n_points, 2))
#     far_lines = np.zeros((n_lines, n_points, 2))

#     for i in range(n_lines):
#         river = resample_line(rivers[i], n_points)

#         for attempt in range(max_attempts):
#             close_candidate = generate_offset_line(river, offset_distance=close_offset, noise_std=10)
#             if is_valid_line(close_candidate, river, min_distance=4.0):
#                 close_lines[i] = resample_line(close_candidate, n_points)
#                 break
#         else:
#             print(f"Warning: Close street for line {i} may be invalid.")
#             close_lines[i] = resample_line(close_candidate, n_points)

#         for attempt in range(max_attempts):
#             far_candidate = generate_offset_line(river, offset_distance=far_offset, noise_std=6)
#             if is_valid_line(far_candidate, river, min_distance=32.0):
#                 far_lines[i] = resample_line(far_candidate, n_points)
#                 break
#         else:
#             print(f"Warning: Far street for line {i} may be invalid.")
#             far_lines[i] = resample_line(far_candidate, n_points)

#     return close_lines, far_lines

In [ ]:
close, far = generate_street_lines_from_rivers(
    original_lines,
    n_points=64,
    close_offset=40,
    far_offset=120
)

In [ ]:
plot_lines(interpolated_lines, close, far, start=105, end=106, title="Original and Synthetic Lines")

In [ ]:
# np.save(f'../data/preprocessing/interpolated/interpolated_close.npy', close)
# np.save(f'../data/preprocessing/interpolated/interpolated_lines_far.npy', far)

## Min Max Normalization

In [ ]:
interpolated_lines_close = np.load(f'../data/preprocessing/rotated/rotated_lines_close.npy')
interpolated_lines_far = np.load(f'../data/preprocessing/rotated/rotated_lines_far.npy')
interpolated_lines_original = np.load(f'../data/preprocessing/rotated/rotated_lines.npy')

In [ ]:
# LOCAL --> TBC: between 0 and 1 
def normalize_triplets_per_sample(original, close, far, method='max'):
    """
    Normalize each sample I using stacked points from original[I], close[I], far[I].
    method: 'max' -> divide by max L2 distance from centroid
            'std' -> divide by standard deviation (per-dim combined -> scalar)
    Returns normalized arrays and arrays of centroids/scales for possible inversion.
    """
    assert original.shape == close.shape == far.shape
    N = original.shape[0]

    orig_norm = np.empty_like(original, dtype=np.float32)
    close_norm = np.empty_like(close, dtype=np.float32)
    far_norm   = np.empty_like(far, dtype=np.float32)

    centroids = np.zeros((N, 2), dtype=np.float32)
    scales = np.zeros((N,), dtype=np.float32)

    for i in range(N):
        stacked = np.vstack([original[i], close[i], far[i]])  # (192, 2)
        centroid = stacked.mean(axis=0)                       # (2,)
        centered = stacked - centroid                        # (192, 2)

        if method == 'max':
            # scale = max L2 distance of any point from centroid
            dists = np.linalg.norm(centered, axis=1)
            scale = dists.max()
        elif method == 'std':
            # scalar std across both dims
            scale = centered.std()
        else:
            raise ValueError("method must be 'max' or 'std'")

        if scale == 0 or np.isnan(scale):
            scale = 1.0

        # unstack
        orig_norm[i]  = (original[i] - centroid) / scale
        close_norm[i] = (close[i] - centroid) / scale
        far_norm[i]   = (far[i] - centroid) / scale

        centroids[i] = centroid
        scales[i] = scale

    return orig_norm, close_norm, far_norm

normalized_original, normalized_synthetic_1, normalized_synthetic_2 = normalize_triplets_per_sample(interpolated_lines_original, interpolated_lines_close, interpolated_lines_far)

In [ ]:
#NEW GLOBAL
combined = np.vstack(interpolated_lines_original)
global_min = combined.min(axis=(0, 1))
global_max = combined.max(axis=(0, 1))

original = np.array(interpolated_lines_original)
close    = np.array(interpolated_lines_close)
far      = np.array(interpolated_lines_far)

normalized_original = (original - global_min) / (global_max - global_min)

# normalized_original = [
#     (line - global_min) / (global_max - global_min)
#     for line in original
# ]

normalized_synthetic_1 = (close - global_min) / (global_max - global_min)
normalized_synthetic_2 = (far - global_min) / (global_max - global_min)

In [ ]:
print(normalized_original.min(), normalized_original.max())
print(np.abs(normalized_original).mean())

In [ ]:
for i in range(5):
    start=i
    plot_lines(normalized_original, normalized_original, normalized_original, start=start, end=start+1, title="Normalized Lines")

In [ ]:
np.save(f'../data/preprocessing/normalized/normalized_global_close.npy', normalized_synthetic_1)
np.save(f'../data/preprocessing/normalized/normalized_global_far.npy', normalized_synthetic_2)
np.save(f'../data/preprocessing/normalized/normalized_global_original.npy', normalized_original)

## Save Results

In [ ]:
# interpolated_lines_close = np.load(f'../data/preprocessing/normalized/normalized_local_close.npy')
# interpolated_lines_far = np.load(f'../data/preprocessing/normalized/normalized_local_far.npy')
# interpolated_lines_original = np.load(f'../data/preprocessing/normalized/normalized_local_original.npy')

np.save(f'../data/preprocessing/normalized/normalized_global_norotation_close.npy', normalized_synthetic_1)
np.save(f'../data/preprocessing/normalized/normalized_global_norotation_far.npy', normalized_synthetic_2)
np.save(f'../data/preprocessing/normalized/normalized_global_norotation_original.npy', normalized_original)

## SINE CURVES

In [ ]:
# Step 1: Generate base X and Y
n_points = 30000
x = np.linspace(0, 1000, n_points)

# Meandering base Y using sine waves + smooth noise
y_main = (
    40 * np.sin(x / 50) +
    20 * np.sin(x / 100) +
    10 * np.sin(x / 200)
)

# Add smoothed base noise to y_main (before normalization)
np.random.seed(42)
base_y_noise = np.random.normal(scale=4, size=n_points)
y_main += gaussian_filter1d(base_y_noise, sigma=150)

# Step 2: Second line (offset version of y)
offset = 20
y_secondary = y_main + offset

# Step 3: Add Gaussian noise to X and Y (before normalization)
x_noise_scale = 0.001
y_noise_scale = 0.5

x_noisy = x + np.random.normal(scale=x_noise_scale, size=n_points)
y_main_noisy = y_main + np.random.normal(scale=y_noise_scale, size=n_points)
y_secondary_noisy = y_secondary + np.random.normal(scale=y_noise_scale, size=n_points)

# Step 4: Normalize both lines to [0, 1] (before adding outliers)
combined_coords = np.vstack((
    np.column_stack((x_noisy, y_main_noisy)),
    np.column_stack((x_noisy, y_secondary_noisy))
))

scaler = MinMaxScaler()
scaler.fit(combined_coords)

normalized_main = scaler.transform(np.column_stack((x_noisy, y_main_noisy)))
normalized_secondary = scaler.transform(np.column_stack((x_noisy, y_secondary_noisy)))

# Step 5: Inject occasional large Y outliers AFTER normalization
outlier_fraction = 0.0005  # 0.05% of points
outlier_magnitude = 0.1    # smaller magnitude because data is now in [0,1]

n_outliers = int(n_points * outlier_fraction)
outlier_indices = np.random.choice(n_points, n_outliers, replace=False)

# Add outliers only to the Y coordinate, scaled for normalized data range
normalized_main[outlier_indices, 1] += np.random.choice([-1, 1], size=n_outliers) * outlier_magnitude
normalized_secondary[outlier_indices, 1] += np.random.choice([-1, 1], size=n_outliers) * outlier_magnitude

# Optional: clip to [0,1] after outlier injection
normalized_main[:, 1] = np.clip(normalized_main[:, 1], 0, 1)
normalized_secondary[:, 1] = np.clip(normalized_secondary[:, 1], 0, 1)

# Plot the entire dataset (or adjust range for sampling)
plot_start = 0
plot_end = 500000

normalized_main_noisy = normalized_main
normalized_secondary_noisy = normalized_secondary

plt.figure(figsize=(12, 5))
plt.plot(normalized_main[plot_start:plot_end, 0], normalized_main[plot_start:plot_end, 1], label='Main Line', color='#2266aa', linewidth=1)
plt.plot(normalized_secondary[plot_start:plot_end, 0], normalized_secondary[plot_start:plot_end, 1], label='Secondary Line', alpha=0.7, color='#7fbfff', linewidth=0.75)
plt.title("Normalized Meandering River Lines with Gaussian Noise and Post-Normalization Y Outliers")
plt.xlabel("X (normalized)")
plt.ylabel("Y (normalized)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# segment_length = 20

# # Trim the data to make sure it divides evenly
# n_segments = normalized_main.shape[0] // segment_length
# trimmed_data = normalized_main[:n_segments * segment_length]

# # Reshape into segments
# curve1 = trimmed_data.reshape(n_segments, segment_length, 2)

# # Trim the data to make sure it divides evenly
# n_segments = normalized_secondary.shape[0] // segment_length
# trimmed_data = normalized_secondary[:n_segments * segment_length]

# # Reshape into segments
# curve2 = trimmed_data.reshape(n_segments, segment_length, 2)

In [ ]:
#np.save(f'/content/drive/MyDrive/ma_files/data/synthetic/synthetic_data_a_norm_noisy.npy', normalized_main_noisy)
#np.save(f'/content/drive/MyDrive/ma_files/data/synthetic/synthetic_data_b_norm_{offset}_noisy.npy', normalized_secondary_noisy)

## CLEAN DATA

In [ ]:
# --- Step 6: Generate clean (ground truth) normalized dataset (no noise, no outliers) ---

# Clean secondary line (just offset)
y_secondary_clean = y_main + offset

# Use the same x (no noise), y_main (no noise), y_secondary_clean (no noise)
normalized_main = scaler.transform(np.column_stack((x, y_main)))
normalized_secondary = scaler.transform(np.column_stack((x, y_secondary_clean)))

print(normalized_main.shape)
print(normalized_secondary.shape)

# Plot the entire dataset (or adjust range for sampling
plot_start = 0
plot_end = 500000

plt.figure(figsize=(12, 5))
plt.plot(normalized_main_noisy[plot_start:plot_end, 0], normalized_main_noisy[plot_start:plot_end, 1], label='Main Line Noisy', color='#009FE3', linewidth=1)
plt.plot(normalized_main[plot_start:plot_end, 0], normalized_main[plot_start:plot_end, 1], label='Main Line', color='#00305D', linewidth=1)
plt.plot(normalized_secondary_noisy[plot_start:plot_end, 0], normalized_secondary_noisy[plot_start:plot_end, 1], label='Secondary Line Noisy', alpha=0.7, color='#FFC000', linewidth=0.75)
plt.plot(normalized_secondary[plot_start:plot_end, 0], normalized_secondary[plot_start:plot_end, 1], label='Secondary Line', alpha=0.7, color='#65B32E', linewidth=0.75)
plt.title("Normalized Meandering River Lines with Gaussian Noise and Post-Normalization Y Outliers")
plt.xlabel("X (normalized)")
plt.ylabel("Y (normalized)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# # Step 1: Generate base X and Y
# n_points = 500000
# x = np.linspace(0, 1000, n_points)

# # Meandering base Y using sine waves + smooth noise
# y_main = (
#     40 * np.sin(x / 50) +
#     20 * np.sin(x / 100) +
#     10 * np.sin(x / 200)
# )

# # Step 2: Second line (offset version of y)
# offset = 20
# y_secondary = y_main + offset

# # Step 4: Normalize both lines to [0, 1] (before adding outliers)
# combined_coords = np.vstack((
#     np.column_stack((x, y_main)),
#     np.column_stack((x, y_secondary))
# ))

# scaler = MinMaxScaler()
# scaler.fit(combined_coords)

# normalized_main = scaler.transform(np.column_stack((x, y_main)))
# normalized_secondary = scaler.transform(np.column_stack((x, y_secondary)))

In [ ]:
# segment_length = 20

# # Trim the data to make sure it divides evenly
# n_segments = normalized_main.shape[0] // segment_length
# trimmed_data = normalized_main[:n_segments * segment_length]

# # Reshape into segments
# curve1 = trimmed_data.reshape(n_segments, segment_length, 2)

# # Trim the data to make sure it divides evenly
# n_segments = normalized_secondary.shape[0] // segment_length
# trimmed_data = normalized_secondary[:n_segments * segment_length]

# # Reshape into segments
# curve2 = trimmed_data.reshape(n_segments, segment_length, 2)

In [ ]:
np.save(f'/content/drive/MyDrive/ma_files/data/synthetic/synthetic_data_a_norm.npy', normalized_main)
np.save(f'/content/drive/MyDrive/ma_files/data/synthetic/synthetic_data_b_norm_{offset}.npy', normalized_secondary)

## STRAIGHT LINE

In [ ]:
num_segments = 10000
points_per_segment = 20
dims = 2

# Total points along the full line
total_points = num_segments * points_per_segment

# Create evenly spaced points along the line from 0 to 1
line_points = np.linspace(0, 1, total_points)

# Since it's a straight line, x and y coordinates are the same
# So points look like [[0,0], [p, p], [q, q], ...]
line_2d = np.stack([line_points, line_points], axis=1)  # shape: [total_points, 2]

# Now reshape into segments of 20 points each
line_segments = line_2d.reshape(num_segments, points_per_segment, dims)  # shape: [10000, 20, 2]

print(line_segments.shape)  # Should print: (10000, 20, 2)

In [ ]:
# Plot for visual check
plt.figure(figsize=(8, 6))
plt.plot(line_segments[:, 0], line_segments[:, 1], alpha=0.7, linewidth=0.7)
plt.title('Straight Line')
plt.xlabel('X')
plt.ylabel('Y')
plt.axis('auto')
plt.show()

In [ ]:
np.save(f'/content/drive/MyDrive/ma_files/data/straight_line.npy', line_segments)